# 14.7 · 时序 CNN / TCN / Temporal Convolutional Networks

> **课程定位 / Where this fits**
> 第 7 课，**Part 14 · 时间序列**。用卷积(而非循环)做时序预测。
> Lesson 7, **Part 14 · Time Series**. Forecasting with convolutions instead of recurrence.
>
> RNN/LSTM 处理序列必须**逐步串行**(慢)。**TCN(时序卷积网络)** 换个思路: 用**一维卷积**处理时序, 可以**完全并行**(训练快得多), 同时靠两个关键设计胜任时序: ①**因果卷积(causal)** —— 每个输出只依赖**当前及过去**(绝不偷看未来);②**空洞卷积(dilated)** —— 卷积核带"间隔"地跳着取样, 让**感受野随层数指数级增大**, 少数几层就能覆盖很长的历史。TCN 在很多时序任务上与 LSTM 相当甚至更好, 且更快。本课从零理解因果/空洞卷积并搭一个 TCN 预测器。
> RNN/LSTM must process sequences **step by step serially** (slow). A **TCN (Temporal Convolutional Network)** uses **1D convolutions** instead — **fully parallelizable** (much faster training) — with two key designs: ① **causal convolution** — each output depends only on **current and past** (never peeks at the future); ② **dilated convolution** — the kernel samples with "gaps," making the **receptive field grow exponentially with depth**, so a few layers cover long history. TCNs match or beat LSTMs on many tasks and are faster. We understand causal/dilated convolution from scratch and build a TCN forecaster.
>
> 💼 **实战/面试视角**："因果卷积怎么保证不看未来 / 空洞卷积如何扩大感受野 / TCN vs RNN(并行/长依赖)" 是深度时序常考。
> 💼 **Practical/interview angle:** "how causal conv avoids the future / how dilation grows the receptive field / TCN vs RNN (parallel/long-range)" — common.

> 📐 **符号约定 / Notation**
> - 因果卷积 —— 输出 $t$ 只依赖输入 $\le t$ / output t depends only on inputs ≤ t
> - 空洞/膨胀 $d$ —— 卷积取样的间隔 / dilation: gap between sampled inputs
> - 感受野 —— 一个输出"看到"多长的历史 / receptive field

> 💡 **面试相关 / Interview-relevant**
> - "因果卷积是什么/怎么实现(左侧padding)"（出镜率 ★★★★）
> - "空洞卷积如何让感受野指数增长"（★★★★★）
> - "TCN 相比 RNN 的优势(并行/稳定梯度/长依赖)"（★★★★★）
> - "TCN 的感受野怎么算"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解用卷积处理时序的思路与 TCN 的优势。
   Understand convolutional sequence modeling and TCN's advantages.
2. 掌握**因果卷积**(不看未来)的实现。
   Master causal convolution (no future peeking).
3. 理解**空洞卷积**如何让感受野指数级增大。
   Understand how dilation grows the receptive field exponentially.
4. **从零搭 TCN** 预测器。
   Build a TCN forecaster from scratch.

## 目录 / TOC
1. [卷积做时序 + TCN 优势 ⭐](#1)
2. [因果卷积:不偷看未来 ⭐](#2)
3. [空洞卷积:指数级感受野 ⭐](#3)
4. [搭 TCN 预测 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 卷积做时序 + TCN 优势 ⭐ / Conv for Sequences & TCN Advantages

CNN 在图像上很成功(Part 10), 它也能处理序列——用**一维卷积(Conv1d)** 沿时间轴滑动, 提取局部时序模式(类似 10.2 的卷积核, 只是在 1D 上)。
CNNs excel on images (Part 10) and also handle sequences — a **1D convolution (Conv1d)** slides along the time axis, extracting local temporal patterns (like 10.2's kernels, but in 1D).

**TCN vs RNN(面试核心对比)**:
**TCN vs RNN (interview core):**
- **并行 vs 串行**:RNN 必须等前一步算完才能算下一步(串行, 慢);TCN 的卷积**所有位置同时算**(并行, 训练快得多)。
  **Parallel vs serial:** RNN waits for the previous step (serial, slow); TCN's convolutions compute **all positions at once** (parallel, much faster training).
- **梯度更稳**:TCN 没有 RNN 的沿时间反向传播(BPTT), 梯度路径更短, **不易梯度消失/爆炸**(呼应 12.1)。
  **Stabler gradients:** no BPTT over time; shorter gradient paths, **less vanishing/exploding** (echoing 12.1).
- **灵活的感受野**:靠空洞卷积可控地覆盖很长历史。
  **Flexible receptive field:** dilation covers long history controllably.

但有两个特殊要求: ①不能让卷积"看到未来"(→因果卷积);②要高效覆盖长历史(→空洞卷积)。下面逐个解决。
But two special needs: ① conv mustn't "see the future" (→ causal conv); ② cover long history efficiently (→ dilated conv). We tackle each.


<a id="2"></a>
## 2. 因果卷积:不偷看未来 ⭐ / Causal Convolution

普通卷积是"**对称**"的: 输出位置 $t$ 由输入 $t$ **左右两侧**的值算出。但时序预测里, 预测 $t$ 时**绝不能用到 $t$ 之后**的信息(那是未来, 等于作弊/泄漏)。
A normal convolution is **symmetric**: output at $t$ uses inputs on **both sides** of $t$. But in forecasting, predicting $t$ must **never use information after $t$** (the future — cheating/leakage).

**因果卷积(causal convolution)** 保证"输出 $t$ 只依赖输入 $\le t$"。实现很简单(面试点): **只在序列左边(过去方向)补零(padding)**, 卷积完再把右边多出来的部分裁掉——这样每个输出位置都只"看左边"。
**Causal convolution** ensures "output $t$ depends only on inputs ≤ $t$." The implementation is simple: **zero-pad only on the left (past side)**, then crop the extra on the right — so each output only "looks left."

下面从零实现因果卷积并**验证它真的不看未来**(改动未来的输入, 当前输出不变)。
We implement causal convolution from scratch and **verify it ignores the future** (changing a future input doesn't change the current output).


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, time
import torch, torch.nn as nn
import warnings; warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid"); torch.manual_seed(0)
ap=[112,118,132,129,121,135,148,148,136,119,104,118,115,126,141,135,125,149,170,170,158,133,114,140,145,150,178,163,172,178,199,199,184,162,146,166,171,180,193,181,183,218,230,242,209,191,172,194,196,196,236,235,229,243,264,272,237,211,180,201,204,188,235,227,234,264,302,293,259,229,203,229,242,233,267,269,270,315,364,347,312,274,237,278,284,277,317,313,318,374,413,405,355,306,271,306,315,301,356,348,355,422,465,467,404,347,305,336,340,318,362,348,363,435,491,505,404,359,310,337,360,342,406,396,420,472,548,559,463,407,362,405,417,391,419,461,472,535,622,606,508,461,390,432]
ts=np.array(ap,dtype=float)

class CausalConv1d(nn.Module):
    """因果卷积: 只在左侧补零, 保证输出t只依赖输入<=t / causal conv: left-pad only."""
    def __init__(self, cin, cout, kernel, dilation=1):
        super().__init__()
        self.pad = (kernel - 1) * dilation               # 左侧需要补的零数(=感受野-1) / left padding
        self.conv = nn.Conv1d(cin, cout, kernel, dilation=dilation)
    def forward(self, x):                                 # x: (batch, channels, time)
        x = nn.functional.pad(x, (self.pad, 0))           # 只在左边(过去)补零 / pad LEFT only
        return self.conv(x)

# 验证因果性: 改动"未来"的输入, 当前及之前的输出应完全不变 / verify causality
cc = CausalConv1d(1, 1, kernel=3)
x = torch.randn(1, 1, 10)
y1 = cc(x)
x2 = x.clone(); x2[0,0,7:] = 999                          # 篡改第7步及之后(未来) / corrupt the future (t>=7)
y2 = cc(x2)
print("篡改未来(t>=7)后, 各输出位置是否改变:")
print((y1 - y2).abs().squeeze().detach().numpy().round(3))
print("→ 前面位置(t<7)的输出完全不变(差=0): 因果卷积确实不看未来 ✓")
print("实现: 只在序列左侧补零(过去方向), 输出t只依赖输入<=t; 这是时序卷积防泄漏的关键")


<a id="3"></a>
## 3. 空洞卷积:指数级感受野 ⭐ / Dilated Convolution

时序常有**长依赖**(去年同月影响今年)。要让卷积"看到"很久以前, 普通做法要么**堆很多层**(每层只扩大感受野一点点, 太深), 要么**用很大的卷积核**(参数爆炸)。**空洞卷积(dilated convolution)** 巧妙解决: 卷积核取样时**带固定间隔 $d$ 地跳着取**(不是连续相邻), 一下子覆盖更宽的范围。
Series often have **long dependencies** (last year's same month affects this year). To make conv "see" far back, naive options are stacking **many layers** (each grows the receptive field a little, too deep) or a **huge kernel** (parameter explosion). **Dilated convolution** solves it elegantly: the kernel **samples with a fixed gap $d$** (not adjacent), covering a wider span at once.

**关键(面试)**: 把膨胀率 $d$ 设成**逐层翻倍**(1, 2, 4, 8, …), **感受野随层数指数级增长**——只要 $O(\log n)$ 层就能覆盖长度 $n$ 的历史! 例如核大小 2、膨胀 1,2,4,8 的 4 层, 感受野 = $1 + 1+2+4+8 = 16$。既高效又能抓长依赖。
**Key (interview):** set dilation $d$ to **double each layer** (1, 2, 4, 8, …) and the **receptive field grows exponentially** — only $O(\log n)$ layers cover history of length $n$! E.g. kernel 2 with dilations 1,2,4,8 over 4 layers → receptive field = $1 + 1+2+4+8 = 16$. Efficient *and* long-range.


In [ ]:
# 感受野随膨胀率翻倍而指数增长 / receptive field grows exponentially with doubling dilation
kernel = 2
print(f"卷积核大小 = {kernel}, 膨胀率逐层翻倍:")
rf = 1
for layer, d in enumerate([1, 2, 4, 8, 16]):
    rf += (kernel - 1) * d                                # 每层增加 (k-1)*d / each layer adds (k-1)*d
    print(f"  第{layer+1}层 (膨胀d={d:2}): 累积感受野 = {rf} 个时间步")
print("→ 仅 5 层就覆盖 32 步历史; 感受野指数增长, 用O(log n)层覆盖长度n的依赖")

# 可视化空洞卷积的取样模式 / visualize dilated sampling pattern
fig, axes = plt.subplots(3, 1, figsize=(11, 5))
for ax, d in zip(axes, [1, 2, 4]):
    positions = np.arange(20)
    ax.plot(positions, np.zeros_like(positions), "o", color="lightgray", ms=8)
    sampled = [19 - i*d for i in range(kernel)]           # 输出最后位置看的输入点 / inputs the last output sees
    ax.plot(sampled, np.zeros(len(sampled)), "ro", ms=12)
    ax.set_title(f"膨胀 d={d}: 卷积核跳着取样(红点), 间隔={d}"); ax.set_yticks([]); ax.set_xlim(-1, 20)
plt.tight_layout(); plt.show()
print("空洞卷积: 卷积核带间隔d取样(d越大跨度越宽); 逐层翻倍d → 感受野指数增长, 高效抓长依赖")


<a id="4"></a>
## 4. 搭 TCN 预测 + 小结 ⭐ / Build a TCN Forecaster

把**因果 + 空洞卷积**堆起来(配残差连接, 呼应 10.4), 就是 **TCN**。下面搭一个小 TCN(膨胀 1,2,4)在航空数据上预测。
Stack **causal + dilated convolutions** (with residual connections, echoing 10.4) to form a **TCN**. Below we build a small TCN (dilations 1,2,4) and forecast the airline data.


In [ ]:
idx = pd.date_range("1949-01", periods=len(ts), freq="MS")
train, test = ts[:120], ts[120:]
mn, mx = train.min(), train.max()                         # 缩放只用训练集(防泄漏, 见14.6) / scale on train only
scale = lambda x: (x-mn)/(mx-mn); unscale = lambda x: x*(mx-mn)+mn
L = 24; s_train = scale(train)
X, Y = [], []
for i in range(len(s_train)-L): X.append(s_train[i:i+L]); Y.append(s_train[i+L])
X = torch.tensor(np.array(X), dtype=torch.float32).unsqueeze(1)   # (样本, 1通道, 时间L) / (N, 1, L)
Y = torch.tensor(np.array(Y), dtype=torch.float32).unsqueeze(-1)

class TCNBlock(nn.Module):
    def __init__(self, c, d):
        super().__init__(); self.c1 = CausalConv1d(c, c, 3, dilation=d); self.c2 = CausalConv1d(c, c, 3, dilation=d); self.relu = nn.ReLU()
    def forward(self, x): return self.relu(x + self.c2(self.relu(self.c1(x))))   # 残差连接 / residual
class TCN(nn.Module):
    def __init__(self, c=16):
        super().__init__()
        self.inp = CausalConv1d(1, c, 1)
        self.blocks = nn.ModuleList([TCNBlock(c, d) for d in [1, 2, 4, 8]])      # 膨胀逐层翻倍 / doubling dilation
        self.head = nn.Linear(c, 1)
    def forward(self, x):
        h = self.inp(x)
        for b in self.blocks: h = b(h)
        return self.head(h[:, :, -1])                     # 取最后时间步预测下一个 / last step → next

torch.manual_seed(0); net = TCN(); opt = torch.optim.Adam(net.parameters(), 5e-3); t0 = time.time()
for ep in range(300): opt.zero_grad(); loss = nn.functional.mse_loss(net(X), Y); loss.backward(); opt.step()
net.eval(); window = list(s_train[-L:]); preds = []
with torch.no_grad():
    for _ in range(len(test)):
        x = torch.tensor(window[-L:], dtype=torch.float32).view(1,1,L); p = net(x).item(); preds.append(p); window.append(p)
fc = unscale(np.array(preds)); mape = np.mean(np.abs((test-fc)/test))*100
fig, ax = plt.subplots(figsize=(11,4))
ax.plot(idx[:120], train, label="训练"); ax.plot(idx[120:], test, label="真实", color="green")
ax.plot(idx[120:], fc, "r--", label="TCN 预测")
ax.legend(); ax.set_title(f"TCN(因果+空洞卷积)递归预测: MAPE={mape:.1f}% (训练{time.time()-t0:.0f}s)")
plt.tight_layout(); plt.show()
print(f"TCN 预测 MAPE = {mape:.1f}% (小数据上同样比不过经典法, 但比LSTM训练更快/可并行)")
print("TCN = 因果卷积(不看未来) + 空洞卷积(指数感受野) + 残差; 并行训练, 长依赖好, 是RNN的有力替代")


```
卷积做时序: Conv1d沿时间滑动提取局部模式; TCN vs RNN: 并行(快)/梯度稳/感受野灵活
因果卷积: 输出t只依赖输入<=t; 实现=只在左侧(过去)补零再裁右边; 防未来泄漏的关键
空洞卷积: 卷积核带间隔d跳着取样; 膨胀逐层翻倍(1,2,4,8)→感受野指数增长; O(log n)层覆盖长度n
TCN = 因果卷积+空洞卷积+残差连接; 并行训练比RNN快, 长依赖好, 很多任务≥LSTM
小数据上(本例)仍比不过经典统计法(深度学习数据饥渴, 同14.6结论)
```

### 💡 面试速查 / Interview cheat-sheet
1. **因果卷积**: 输出t只看输入≤t; 左侧padding实现; 防未来泄漏。
   Causal conv: output t sees only inputs ≤ t; left-padding; prevents future leakage.
2. **空洞卷积**: 核带间隔d取样; 膨胀翻倍→感受野指数增长(O(log n)层覆盖长依赖)。
   Dilated conv: gapped sampling; doubling dilation → exponential receptive field.
3. **TCN vs RNN**: 卷积并行(训练快)+梯度稳+长依赖; 很多任务与LSTM相当或更好。
   TCN vs RNN: parallel (fast) + stable gradients + long-range; matches/beats LSTM often.
4. **TCN结构**: 因果+空洞卷积+残差连接堆叠。
   TCN: causal + dilated conv + residuals, stacked.
5. **感受野**: 1+Σ(k-1)·d_layer; 膨胀翻倍时指数增长。
   Receptive field: 1+Σ(k-1)·d; exponential with doubling dilation.

### 下一节 / Next
**14.8 Transformer 预测**——把 Part 12 的 Transformer 用到时序。注意力能直接建模任意距离依赖, 但原始 Transformer 对长序列贵($O(n^2)$)、且需特别设计。我们会讲 **Informer、PatchTST** 等时序 Transformer 的核心思想(如把序列切成 patch)。
**14.8 Transformers for TS** — apply Part 12's Transformer to time series. Attention models any-distance dependencies directly, but vanilla Transformers are costly for long sequences ($O(n^2)$) and need special design. We'll cover the core ideas of TS Transformers like **Informer, PatchTST** (e.g. patching the series).
